# MSIS 522 HW1: Yelp High-Rating Classification

This notebook is the grading document for the end-to-end workflow implemented in this repository. The prediction task is binary classification: identify whether a Yelp business is likely to be **high-rated** (`stars >= 4`).

Why this matters: rating quality is a practical business signal. It captures customer satisfaction, reputation, and long-run competitiveness. A model that explains which business attributes are associated with stronger ratings can help operators prioritize decisions around location, operations, and service features.

## 1. Reproducibility

Run the full pipeline from the repository root:

```bash
python -m src.pipeline.run_all --skip-download
streamlit run app/streamlit_app.py
```

The app reads pre-trained artifacts only; it does not retrain models at runtime.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
ARTIFACTS = ROOT / 'artifacts'

summary = json.loads((ARTIFACTS / 'data' / 'dataset_summary.json').read_text(encoding='utf-8'))
metrics_path = ARTIFACTS / 'metrics' / 'model_metrics.csv'
metrics = pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()
summary


## 2. Dataset Introduction

- **Source**: Yelp Academic Dataset, business table only.
- **Target**: `high_rating = 1 if stars >= 4 else 0`.
- **Unit of analysis**: one row per business.
- **Feature mix**: operational variables (`is_open`, hours-derived features), engagement (`review_count`), geography (`city`, `state`), and engineered business-category / attribute indicators.

This framing creates a clear and interpretable tabular classification problem that matches the assignment requirements well.

In [ ]:
dataset_overview = pd.DataFrame([
    {
        'Rows': summary['n_rows'],
        'Features': summary['n_features'],
        'Numerical Features': summary['numeric_feature_count'],
        'Categorical Features': summary['categorical_feature_count'],
        'Negative Class (0)': summary['class_distribution']['0'],
        'Positive Class (1)': summary['class_distribution']['1'],
    }
])
dataset_overview


## 3. Descriptive Analytics

The following visuals correspond directly to Part 1 of the rubric. Each one is paired with a concise interpretation that can be reused in the app or submission notes.

In [ ]:
figure_files = [
    'target_distribution.png',
    'review_count_distribution_by_target.png',
    'review_count_boxplot_by_target.png',
    'high_rating_share_by_state_top15.png',
    'high_rating_share_by_top_categories.png',
    'correlation_heatmap.png',
]

for fig_name in figure_files:
    fig_path = ARTIFACTS / 'figures' / fig_name
    if fig_path.exists():
        plt.figure(figsize=(12, 6))
        plt.imshow(plt.imread(fig_path))
        plt.axis('off')
        plt.title(fig_name)
        plt.show()


### Visual Interpretation

**Target distribution**: the classes are close to balanced, which is helpful for classification. This reduces the risk that headline metrics are inflated by a dominant class and makes F1 and ROC-AUC more meaningful.

**Review-count histogram**: customer engagement is highly skewed, with a long right tail. A small fraction of businesses collects very large review volumes, suggesting that log-scale thinking or robust models matters more than naive linear assumptions.

**Review-count boxplot**: the class-wise spread is wide, and outliers remain visible even on a log scale. This suggests that review volume alone is not sufficient, but it is likely to be an informative predictor when combined with category and operational features.

**State-level high-rating share**: some geographies show a clearly higher share of highly rated businesses. That pattern implies location carries useful signal, though it may also proxy for unobserved market or customer-behavior differences.

**Category-level high-rating share**: some business types are structurally more likely to receive strong ratings. This is important because it shows why engineered category indicators should improve tree-based models.

**Correlation heatmap**: the strongest correlations are concentrated among operationally related variables rather than everything moving together. That is a good sign for modeling because it suggests useful multi-feature signal without extreme redundancy everywhere.

## 4. Predictive Analytics

The evaluation setup uses a stratified 70/30 train-test split with `random_state=42`. Hyperparameter tuning uses cross-validation on the training split, and final metrics are reported on the held-out test set.

Required models:
- Logistic Regression baseline
- Decision Tree (GridSearchCV)
- Random Forest (GridSearchCV)
- XGBoost (GridSearchCV)
- MLP neural network


In [ ]:
metrics.sort_values('f1', ascending=False) if not metrics.empty else 'Run training first to populate model_metrics.csv.'


### Model Comparison Discussion

In the current run, **XGBoost** is the strongest predictive model with **F1 = 0.683** and **ROC-AUC = 0.753**. That result is materially better than the Logistic Regression baseline (**F1 = 0.641**), which indicates that nonlinear interactions between geography, business category, operating hours, and review activity matter for this Yelp classification problem.

The **PyTorch MLP** is the runner-up (**F1 = 0.665**), which is useful because it shows a neural network can compete on this tabular problem, but it still does not beat the best tree ensemble. In practice, the tradeoff is that XGBoost and Random Forest deliver stronger predictive accuracy, while Logistic Regression and the single Decision Tree are easier to interpret directly.

For the write-up, the key business point is simple: the best-performing model is not the simplest model. That suggests stakeholders should care about interacting signals rather than single-variable stories.


In [ ]:
comparison_figures = [
    'model_comparison_f1.png',
    'roc_curves_all_models.png',
    'dt_best_tree.png',
    'mlp_training_history.png',
    'mlp_tuning_results.png',
]

for fig_name in comparison_figures:
    fig_path = ARTIFACTS / 'figures' / fig_name
    if fig_path.exists():
        plt.figure(figsize=(12, 6))
        plt.imshow(plt.imread(fig_path))
        plt.axis('off')
        plt.title(fig_name)
        plt.show()


## 5. Explainability (SHAP)

SHAP is applied to the strongest available tree-based explainer path in the saved artifacts. The predictive winner is XGBoost, but in this environment SHAP encounters an XGBoost parsing issue, so the saved explanation artifacts fall back to the **Random Forest** model. That is acceptable for interpretation because Random Forest is still one of the top-performing tree models and remains faithful to the assignment requirement of using a tree-based model.

Interpretation should focus on three questions:
1. Which features move predictions the most?
2. Do those features push the model toward or away from the positive class?
3. How could a stakeholder act on those insights in the Yelp business context?

In this project, the strongest SHAP drivers are primarily operating intensity, review engagement, and business category indicators. That gives a manager something actionable: understand whether a business profile resembles other highly rated businesses in both operations and customer traction, not just in one isolated attribute.


In [ ]:
for fig_name in ['summary.png', 'bar.png', 'waterfall_example.png']:
    fig_path = ARTIFACTS / 'shap' / fig_name
    if fig_path.exists():
        plt.figure(figsize=(12, 6))
        plt.imshow(plt.imread(fig_path))
        plt.axis('off')
        plt.title(fig_name)
        plt.show()

interpretation_path = ARTIFACTS / 'shap' / 'interpretation.md'
if interpretation_path.exists():
    print(interpretation_path.read_text(encoding='utf-8'))


## 6. Streamlit Deployment

The submitted app must contain four tabs:
- Executive Summary
- Descriptive Analytics
- Model Performance
- Explainability & Interactive Prediction

Before submission, replace the placeholder below with your public deployed URL from Streamlit Community Cloud.

**Deployment URL:** `ADD_PUBLIC_STREAMLIT_URL_HERE`

Final verification: open the deployed link in an incognito window and confirm that all four tabs load, the figures render, and the interactive prediction section works without retraining.
